<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week03_transformers/week03_tiny_transformer.ipynb)

# Building a Decoder-Only Transformer from Scratch on FOMC Text

**TAE Week 3 Capstone** 

This notebook builds and trains a small decoder-only Transformer from scratch in PyTorch on FOMC statements and minutes from 2010 through April 2026. Scaled dot-product attention, causal masking, multi-head attention, positional encoding, Transformer blocks, training, checkpointing, and autoregressive generation are implemented directly, without Transformer libraries.

The core model uses character-level tokenization. A separate BPE extension tests how tokenization changes language-model efficiency under the same raw-text train/validation split, Transformer depth and width, block size, and token-context budget. Final comparison uses deterministic unsmoothed held-out negative log-likelihood normalized to bits per character.

## Setup

In [ ]:
!nvidia-smi || true
!pip -q install tqdm


In [ ]:
import math, json, time, hashlib
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

plt.style.use('seaborn-v0_8')
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {DEVICE}")

NOTEBOOK_START = time.time()  # for the total-runtime cell at the very end


## Configuration

In [ ]:
SEED = 1
torch.manual_seed(SEED)
np.random.seed(SEED)

@dataclass
class ModelConfig:
    vocab_size: int = None
    d_model: int = 256
    num_heads: int = 8
    num_layers: int = 6
    d_ff: int = 1024
    block_size: int = 256
    dropout: float = 0.1
    label_smoothing: float = 0.05
    use_fused_attention: bool = True  # training speed; verified equivalent to the
                                       # hand-written attention used in the checks below

@dataclass
class TrainConfig:
    batch_size: int = 64
    lr: float = 3e-4
    min_lr: float = 3e-5
    warmup_iters: int = 200
    weight_decay: float = 0.1   # regularization strength; the BPE extension uses 0.08
                                 # as a hypothesis, not a controlled causal result
    grad_clip: float = 1.0
    max_steps: int = 6000
    eval_interval: int = 300
    eval_iters: int = 50

CORPUS_PATH = Path("fomc_training_corpus.txt")
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / "tiny_transformer_best.pt"
RUN_DIR = Path("runs"); RUN_DIR.mkdir(exist_ok=True)

# TRAIN=True is the fresh-run default. Set False to use local checkpoints when available.
TRAIN = True


## Data

Character-level, a 90/10 train/validation split, and fixed-length context sampling.

In [ ]:
# Running on Colab (or anywhere the corpus file isn't already present)?
# Fetch it directly from this repo so the notebook is a genuine single-click
# run -- no manual upload needed.
if not CORPUS_PATH.exists():
    import subprocess, shutil, tempfile
    print("corpus not found locally -- fetching from GitHub...")
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", "main",
            "https://github.com/FranQuant/the-ai-engineer.git", tmp
        ], check=True, capture_output=True)
        src_dir = Path(tmp) / "capstones" / "week03_transformers"
        shutil.copy(src_dir / "fomc_training_corpus.txt", CORPUS_PATH)
    print(f"fetched: {CORPUS_PATH} ({CORPUS_PATH.stat().st_size:,} bytes)")
else:
    print(f"corpus already present: {CORPUS_PATH} ({CORPUS_PATH.stat().st_size:,} bytes)")


In [ ]:
text = CORPUS_PATH.read_text(encoding="utf-8")
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

n_split = int(0.9 * len(text))
train_text, val_text = text[:n_split], text[n_split:]
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)
assert train_text + val_text == text
print(f"corpus: {len(text):,} chars | vocab: {vocab_size} | train: {len(train_data):,} | val: {len(val_data):,}")

model_cfg = ModelConfig(vocab_size=vocab_size)
train_cfg = TrainConfig()

def get_batch(split, block_size, batch_size, device=DEVICE):
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)


## Scaled Dot-Product Attention

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        S = S.masked_fill(mask == 0, float("-inf"))
    S = S - S.max(dim=-1, keepdim=True).values
    A = torch.softmax(S, dim=-1)
    return A @ V, S, A


A quick check against a hand-computable example: three tokens, 2D vectors.

In [ ]:
Qe = torch.tensor([[1.,0],[0,1],[1,1]])
Ke = torch.tensor([[1.,0],[1,1],[0,1]])
Ve = torch.tensor([[1.,0],[0,2],[3,1]])
Y, S, A = scaled_dot_product_attention(Qe, Ke, Ve)
assert torch.allclose(A.sum(dim=-1), torch.ones(3))
assert torch.allclose(Y, torch.tensor([[0.994440, 1.0], [1.401112, 1.203336], [0.993020, 1.255235]]), atol=1e-5)
print("attention weights sum to 1, output matches hand computation:")
print(Y)

# The training path uses PyTorch's fused SDPA. Verify it against the hand-written
# implementation above with identical inputs and dropout disabled.
_parity_gen = torch.Generator().manual_seed(SEED)
Q_parity = torch.randn(2, 3, 5, 4, generator=_parity_gen)
K_parity = torch.randn(2, 3, 5, 4, generator=_parity_gen)
V_parity = torch.randn(2, 3, 5, 4, generator=_parity_gen)
for _causal in (False, True):
    _mask = torch.tril(torch.ones(5, 5)) if _causal else None
    manual_out, _, _ = scaled_dot_product_attention(Q_parity, K_parity, V_parity, mask=_mask)
    fused_out = F.scaled_dot_product_attention(
        Q_parity, K_parity, V_parity, dropout_p=0.0, is_causal=_causal
    )
    assert torch.allclose(manual_out, fused_out, atol=1e-6, rtol=1e-5)
print("[OK] hand-written and fused SDPA agree for causal=False and causal=True")


With a causal mask, the first token has exactly one valid key -- itself -- so its attention row is forced to `[1, 0, 0]` no matter what the key vectors are.

In [ ]:
causal_mask = torch.tril(torch.ones(3, 3))
_, _, A_m = scaled_dot_product_attention(Qe, Ke, Ve, mask=causal_mask)
assert torch.allclose(A_m[0], torch.tensor([1., 0., 0.]))
assert torch.count_nonzero(torch.triu(A_m, diagonal=1)) == 0
assert torch.allclose(A_m.sum(dim=-1), torch.ones(3))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].imshow(A.detach(), cmap="Blues", vmin=0, vmax=1); axes[0].set_title("unmasked")
axes[1].imshow(A_m.detach(), cmap="Blues", vmin=0, vmax=1); axes[1].set_title("causal")
for ax in axes: ax.set_xticks(range(3)); ax.set_yticks(range(3))
fig.tight_layout(); plt.show()


## Self-Attention

Wraps attention with learned query/key/value projections.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k=None, d_v=None, causal=True):
        super().__init__()
        d_k, d_v = d_k or d_model, d_v or d_model
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)
        self.causal = causal

    def forward(self, x):
        B, T, _ = x.shape
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        mask = torch.tril(torch.ones(T, T, device=x.device)) if self.causal else None
        y, _, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        return y


Sanity check: zero out the query/key projections so every attention score is equal, and set the value projection to identity. Attention then has to be uniform, so the layer collapses to a plain average over positions.

In [ ]:
sa = SelfAttention(d_model=3, causal=False)
with torch.no_grad():
    sa.W_Q.weight.zero_(); sa.W_K.weight.zero_(); sa.W_V.weight.copy_(torch.eye(3))
x_probe = torch.randn(1, 4, 3)
assert torch.allclose(sa(x_probe), x_probe.mean(dim=1, keepdim=True).expand_as(x_probe), atol=1e-5)
print("reduces to a positional average, as expected")


## Multi-Head Attention and Transformer Blocks

Several attention heads run in parallel, each free to specialize on a different kind of relationship between tokens; a feedforward layer and residual connections around both sublayers complete the block.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0, causal=True, use_fused=False):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads, self.d_head = num_heads, d_model // num_heads
        self.causal = causal
        self.use_fused = use_fused  # default False: unit-test cells below use the
                                     # hand-written path they verify against
        self.proj_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj_out = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.proj_qkv(x).view(B, T, 3, self.num_heads, self.d_head)
        Q, K, V = (t.transpose(1, 2) for t in qkv.unbind(dim=2))
        if self.use_fused:
            Y = F.scaled_dot_product_attention(Q, K, V, is_causal=self.causal)
        else:
            mask = torch.tril(torch.ones(T, T, device=x.device)) if self.causal else None
            Y, _, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        Y = Y.transpose(1, 2).contiguous().view(B, T, D)
        return self.dropout(self.proj_out(Y))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                  nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.0, causal=True, use_fused=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout=dropout, causal=causal, use_fused=use_fused)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


With identical parameters across heads, multi-head attention should match single-head attention exactly -- checked here at `num_heads=1`.

In [ ]:
sa_ref = SelfAttention(d_model=4, causal=True)
mha_ref = MultiHeadAttention(d_model=4, num_heads=1, causal=True)
with torch.no_grad():
    mha_ref.proj_qkv.weight.copy_(torch.cat([sa_ref.W_Q.weight, sa_ref.W_K.weight, sa_ref.W_V.weight], dim=0))
    mha_ref.proj_out.weight.copy_(torch.eye(4))
x2 = torch.randn(1, 3, 4)
assert torch.allclose(mha_ref(x2), sa_ref(x2), atol=1e-5)
print("multi-head (1 head) matches single-head attention")


In [ ]:
tiny_block = TransformerBlock(d_model=4, num_heads=2, d_ff=8, causal=True)
x_tiny = torch.randn(1, 3, 4)
out_tiny = tiny_block(x_tiny)
assert out_tiny.shape == (1, 3, 4) and torch.isfinite(out_tiny).all()
print(f"block forward pass ok, shape {tuple(out_tiny.shape)}")


## Positional Encoding

Standard sinusoidal encoding, added to the token embeddings.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.shape[1], :]


## The Model

Token embedding + positional encoding, a stack of transformer blocks, a final LayerNorm, and an output projection tied to the input embedding.

In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.block_size = cfg.block_size
        self.label_smoothing = cfg.label_smoothing
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_enc = PositionalEncoding(cfg.d_model, max_len=cfg.block_size)
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg.d_model, cfg.num_heads, cfg.d_ff, cfg.dropout,
                              causal=True, use_fused=cfg.use_fused_attention)
            for _ in range(cfg.num_layers)
        ])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None):
        z = self.pos_enc(self.tok_emb(idx))
        for blk in self.blocks:
            z = blk(z)
        z = self.ln_f(z)
        logits = self.head(z)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1),
                                    label_smoothing=self.label_smoothing)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, greedy=False, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            if greedy:
                next_id = logits.argmax(dim=-1, keepdim=True)
            else:
                if top_k is not None:
                    k = min(top_k, logits.size(-1))
                    kth_val = torch.topk(logits, k, dim=-1).values[:, -1, None]
                    logits = logits.masked_fill(logits < kth_val, float("-inf"))
                if top_p is not None:
                    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                    cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    remove = cum_probs > top_p
                    remove[..., 1:] = remove[..., :-1].clone()
                    remove[..., 0] = False
                    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
                    logits = torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
                next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = TinyTransformerLM(model_cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")


Before trusting this on the real corpus: can the architecture even memorize a trivial two-character repeating pattern? A model that fails this can't be expected to learn something as complex as FOMC prose.

In [ ]:
torch.manual_seed(SEED)
toy_text = "AB" * 200
toy_stoi = {c: i for i, c in enumerate(sorted(set(toy_text)))}
toy_data = torch.tensor([toy_stoi[c] for c in toy_text])
def toy_batch(bs=8, n=16):
    ix = torch.randint(0, len(toy_data) - bs - 1, (n,))
    return (torch.stack([toy_data[i:i+bs] for i in ix]),
            torch.stack([toy_data[i+1:i+1+bs] for i in ix]))

toy_cfg = ModelConfig(vocab_size=len(toy_stoi), d_model=16, num_heads=2, num_layers=2, d_ff=32,
                       block_size=8, dropout=0.0, label_smoothing=0.0)  # pure capacity check, no regularization
toy_model = TinyTransformerLM(toy_cfg)
toy_opt = torch.optim.Adam(toy_model.parameters(), lr=3e-3)
toy_loss = None
for step in range(300):
    xb, yb = toy_batch()
    _, toy_loss = toy_model(xb, yb)
    toy_opt.zero_grad(); toy_loss.backward(); toy_opt.step()
assert toy_loss.item() < 0.05, toy_loss.item()
print(f"overfits cleanly (final loss {toy_loss.item():.4f})")


## Run Record

A small JSON file capturing what was trained and what it achieved.

In [ ]:
def save_final_run(*, model_name, tokenizer, cfg_model, cfg_train, resume_config,
                   split_metadata, checkpoint_path, parameter_count,
                   initial_best_smoothed_val_ce, resume_best_smoothed_val_ce,
                   final_unsmoothed_nll, final_bpc, evaluation,
                   corpus_sha256, output_dir=RUN_DIR):
    """Write a final record only after deterministic held-out metrics exist."""
    if final_unsmoothed_nll is None or final_bpc is None:
        raise ValueError("final NLL and BPC must come from the completed evaluator")
    timestamp = datetime.now().isoformat()
    rec = {
        "seed": SEED,
        "model_name": model_name,
        "tag": model_name,
        "model_config": asdict(cfg_model),
        "base_train_config": asdict(cfg_train),
        "resume_config": resume_config,
        "corpus_sha256": corpus_sha256,
        "split_metadata": split_metadata,
        "parameter_count": parameter_count,
        "checkpoint_path": str(checkpoint_path),
        "initial_phase_best_smoothed_validation_ce": initial_best_smoothed_val_ce,
        "resume_phase_best_smoothed_validation_ce": resume_best_smoothed_val_ce,
        "final_unsmoothed_nll": final_unsmoothed_nll,
        "final_bpc": final_bpc,
        "evaluation": evaluation,
        "tokenizer": tokenizer,
        "timestamp": timestamp,
    }
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"run_{model_name}_{datetime.now().strftime('%Y%m%dT%H%M%S')}.json"
    path.write_text(json.dumps(rec, indent=2, sort_keys=True), encoding="utf-8")
    return path


## Training (Character-Level)

Adam with a cosine learning-rate schedule, a short warmup, weight decay, and gradient
clipping. Learning rate and the total gradient norm returned immediately before clipping
are recorded at every evaluation step, alongside the loss. Training is guarded by the
`TRAIN` flag. `TRAIN=True` is the honest fresh-run default and generates local checkpoint
artifacts; `TRAIN=False` skips training and loads a checkpoint only when one already exists
locally. Checkpoints are not committed.

In [ ]:
@torch.no_grad()
def evaluate(model, split, block_size, batch_size, iters, get_batch_fn):
    was_training = model.training
    model.eval()
    try:
        losses = [model(*get_batch_fn(split, block_size, batch_size))[1].item() for _ in range(iters)]
    finally:
        model.train(was_training)
    return sum(losses) / len(losses)

@torch.no_grad()
def evaluate_unsmoothed_nll(model, token_ids, block_size, raw_characters_scored, device=DEVICE, stride=None):
    """Score each target once with unsmoothed CE over deterministic sliding windows."""
    if token_ids.ndim != 1 or len(token_ids) < 2:
        raise ValueError("token_ids must be a 1D tensor containing a prefix and at least one target")
    if block_size < 1:
        raise ValueError("block_size must be positive")
    if raw_characters_scored <= 0:
        raise ValueError("raw_characters_scored must be positive")
    stride = max(1, block_size // 2) if stride is None else stride
    if not 1 <= stride <= block_size:
        raise ValueError("stride must be between 1 and block_size")

    was_training = model.training
    model.eval()
    total_nll = torch.zeros((), device=device)
    try:
        # token_ids[0] is a one-token prefix from the training partition, so every
        # held-out token is scored. Windows overlap for context, but scored targets do not.
        n_targets = len(token_ids) - 1
        for target_start in range(0, n_targets, stride):
            target_stop = min(target_start + stride, n_targets)
            context_start = max(0, target_stop - block_size)
            x = token_ids[context_start:target_stop].unsqueeze(0).to(device)
            y = token_ids[context_start + 1:target_stop + 1].unsqueeze(0).to(device)
            logits, _ = model(x)
            targets_to_score = target_stop - target_start
            total_nll += F.cross_entropy(
                logits[:, -targets_to_score:, :].reshape(-1, logits.size(-1)),
                y[:, -targets_to_score:].reshape(-1),
                reduction="sum", label_smoothing=0.0
            )
    finally:
        model.train(was_training)

    return total_nll.item(), raw_characters_scored

def lr_at(step, cfg: TrainConfig):
    if step < cfg.warmup_iters:
        return cfg.lr * step / max(1, cfg.warmup_iters)
    progress = (step - cfg.warmup_iters) / max(1, cfg.max_steps - cfg.warmup_iters)
    return cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * progress))

def save_checkpoint(model, cfg, val_loss, path):
    torch.save({"model_state_dict": model.state_dict(), "model_config": asdict(cfg),
                "val_loss": val_loss}, path)

corpus_sha256 = hashlib.sha256(text.encode("utf-8")).hexdigest()
split_metadata = {
    "type": "contiguous_raw_text_90_10",
    "raw_character_boundary": n_split,
    "train_raw_characters": len(train_text),
    "validation_raw_characters": len(val_text),
}
CHAR_RESUME_STEPS = 1200
CHAR_RESUME_LR = 8e-5
BPE_RESUME_STEPS = 1200
BPE_RESUME_LR = 8e-5
RESUME_EVAL_INTERVAL = 300
RESUME_EVAL_ITERS = 60
EARLY_STOP_PATIENCE = 3
EARLY_STOP_MIN_DELTA = 0.01


In [ ]:
if TRAIN:
    opt = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
    history = {"step": [], "train_loss": [], "val_loss": [], "lr": [], "pre_clip_grad_norm": []}
    best_val = float("inf")
    pbar = tqdm(range(1, train_cfg.max_steps + 1))
    for step in pbar:
        lr = lr_at(step, train_cfg)
        for g in opt.param_groups: g["lr"] = lr
        xb, yb = get_batch("train", model_cfg.block_size, train_cfg.batch_size)
        _, loss = model(xb, yb)
        opt.zero_grad(); loss.backward()
        pre_clip_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        opt.step()
        if step % train_cfg.eval_interval == 0 or step == train_cfg.max_steps:
            tr = evaluate(model, "train", model_cfg.block_size, train_cfg.batch_size, train_cfg.eval_iters, get_batch)
            va = evaluate(model, "val", model_cfg.block_size, train_cfg.batch_size, train_cfg.eval_iters, get_batch)
            history["step"].append(step); history["train_loss"].append(tr); history["val_loss"].append(va)
            history["lr"].append(lr); history["pre_clip_grad_norm"].append(float(pre_clip_grad_norm))
            pbar.set_postfix(train=f"{tr:.3f}", val=f"{va:.3f}", lr=f"{lr:.1e}", pre_gnorm=f"{float(pre_clip_grad_norm):.2f}")
            if va < best_val:
                best_val = va
                save_checkpoint(model, model_cfg, va, CKPT_PATH)


In [ ]:
char_initial_best_val = None
char_resume_best_val = None
if TRAIN:
    char_initial_best_val = best_val
    char_best_val = char_initial_best_val  # kept separate from BPE metrics
    print(f"local checkpoint (initial best val {char_best_val:.4f}): {CKPT_PATH}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history["step"], history["train_loss"], label="train")
    axes[0].plot(history["step"], history["val_loss"], label="val")
    axes[0].set(xlabel="step", ylabel="loss", title="Training and validation loss"); axes[0].legend()
    axes[1].plot(history["step"], history["pre_clip_grad_norm"], color="firebrick")
    axes[1].set(xlabel="step", ylabel="total grad norm (pre-clip)", title="Pre-clipping gradient norm")
    fig.tight_layout(); plt.show()
    print("Loss falls smoothly with no train/val divergence -- no overfitting signal at "
          "this corpus size. Pre-clipping gradient norms settle after an early adjustment "
          "period and stay bounded, consistent with stable optimization.")
else:
    print("TRAIN is False -- character training is skipped; a local checkpoint will be loaded if present.")


### Resume Phase (Character-Level)

Reloads the just-trained checkpoint and continues at a lower, cosine-decaying learning
rate with early stopping -- a second, gentler pass rather than one long first one. Uses
the identical mechanism verified for the BPE model further below, retargeted here rather
than reimplemented.

In [ ]:
if TRAIN:
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    opt_resume = torch.optim.AdamW(model.parameters(), lr=CHAR_RESUME_LR, weight_decay=0.0)
    patience_ctr = 0
    char_resume_history = {"step": [], "val_loss": []}

    pbar = tqdm(range(1, CHAR_RESUME_STEPS + 1))
    for step in pbar:
        progress = step / CHAR_RESUME_STEPS
        lr = CHAR_RESUME_LR * 0.5 * (1 + math.cos(math.pi * progress))
        for g in opt_resume.param_groups: g["lr"] = lr
        xb, yb = get_batch("train", model_cfg.block_size, train_cfg.batch_size)
        _, loss = model(xb, yb)
        opt_resume.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt_resume.step()
        if step % RESUME_EVAL_INTERVAL == 0 or step == CHAR_RESUME_STEPS:
            va = evaluate(model, "val", model_cfg.block_size, train_cfg.batch_size, RESUME_EVAL_ITERS, get_batch)
            char_resume_history["step"].append(step); char_resume_history["val_loss"].append(va)
            pbar.set_postfix(val=f"{va:.3f}", lr=f"{lr:.1e}")
            if va < char_best_val - EARLY_STOP_MIN_DELTA:
                char_best_val = va; patience_ctr = 0
                save_checkpoint(model, model_cfg, va, CKPT_PATH)
            else:
                patience_ctr += 1
                if patience_ctr >= EARLY_STOP_PATIENCE:
                    print(f"early stopping at step {step}: no improvement for {EARLY_STOP_PATIENCE} evals")
                    break

    char_resume_best_val = min(char_resume_history["val_loss"])
    print(f"char-level resume phase best val loss: {char_resume_best_val:.4f}")


### Checkpoint Reload + Smoke Test

In [ ]:
if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    loaded_cfg = ModelConfig(**ckpt["model_config"])
    model = TinyTransformerLM(loaded_cfg).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    char_checkpoint_val = ckpt["val_loss"]
    val_loss_reloaded = evaluate(model, "val", loaded_cfg.block_size, 32, 20, get_batch)
    print(f"local checkpoint loaded; stored best val CE {char_checkpoint_val:.4f}, "
          f"fresh stochastic diagnostic {val_loss_reloaded:.4f}")
else:
    char_checkpoint_val = None
    print(f"no local checkpoint at {CKPT_PATH}; set TRAIN=True and run training to create it")
    val_loss_reloaded = None


## Sampling Gallery (Character-Level)

Greedy decoding, and temperature sampling with `top_k=40` (reduces repetition loops relative to unrestricted sampling).

In [ ]:
def sample(model, prompt, max_new_tokens=200, temperature=1.0, greedy=False, top_k=None, top_p=None):
    model.eval()
    unknown = sorted(set(prompt) - set(stoi))
    if unknown:
        raise ValueError(f"prompt has character(s) outside the training vocabulary: {unknown!r}")
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    return decode(model.generate(idx, max_new_tokens, temperature=temperature, greedy=greedy,
                                  top_k=top_k, top_p=top_p)[0].tolist())

char_prompts = ["<|fomc_statement|>\ndate: 2026-", "The Committee decided to ", "<|fomc_minutes|>\ndate: 2026-"]
char_prompts = [p for p in char_prompts if not (set(p) - set(stoi))]

import textwrap
def show(label, text, width=86, indent=18):
    wrapped = textwrap.fill(text, width=width, subsequent_indent=" " * indent)
    print(f"{label:<{indent}} {wrapped}")

if CKPT_PATH.exists():
    for p in char_prompts:
        print("=" * 92)
        print(f"PROMPT: {p!r}")
        print("-" * 92)
        show("greedy:", sample(model, p, 150, greedy=True))
        for t in (0.7, 1.0, 1.5):
            show(f"t={t:.1f}, top_k=40:", sample(model, p, 150, temperature=t, top_k=40))
        print()


## Final Checks

In [ ]:
print("=== Pre-extension checks ===")
assert torch.allclose(A_m[0], torch.tensor([1., 0., 0.]))
assert torch.count_nonzero(torch.triu(A_m, diagonal=1)) == 0
assert torch.allclose(A_m.sum(dim=-1), torch.ones(3))
print("[OK] causal mask support and row sums verified")
if CKPT_PATH.exists():
    _, loss_check = model(*get_batch("val", model_cfg.block_size, 8))
    assert torch.isfinite(loss_check)
    print(f"[OK] forward pass + loss (val sample: {loss_check.item():.4f})")
    print(f"[OK] local checkpoint: {CKPT_PATH}")
elif TRAIN:
    raise FileNotFoundError(f"TRAIN=True completed without creating {CKPT_PATH}")
else:
    print(f"[INFO] no local character checkpoint; checkpoint-dependent checks were skipped")
assert SEED is not None and model_cfg.vocab_size is not None
print("[OK] config + seed")


## Attention on a Real Prompt

The earlier attention checks used a tiny synthetic example to verify the mechanism is correct. This figure shows what the trained model actually attends to on a real sentence from the corpus: each head's attention weights, visualized as a grid where darker cells mean a given word attends more strongly to an earlier word in the same sentence.

In [ ]:
if CKPT_PATH.exists():
    probe_text = decode(train_data[1000:1030].tolist())
    probe_idx = torch.tensor([encode(probe_text)], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        z = model.blocks[0].ln1(model.pos_enc(model.tok_emb(probe_idx)))
        attn0 = model.blocks[0].attn
        B, T, D = z.shape
        qkv = attn0.proj_qkv(z).view(B, T, 3, attn0.num_heads, attn0.d_head)
        Qp, Kp, Vp = (t.transpose(1, 2) for t in qkv.unbind(dim=2))
        mask = torch.tril(torch.ones(T, T, device=z.device))
        _, _, Ap = scaled_dot_product_attention(Qp, Kp, Vp, mask=mask)
    n_show = min(4, attn0.num_heads)
    fig, axes = plt.subplots(1, n_show, figsize=(3.2 * n_show, 3.2))
    for h in range(n_show):
        ax = axes[h] if n_show > 1 else axes
        ax.imshow(Ap[0, h].cpu(), cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"head {h}"); ax.set_xticks(range(T)); ax.set_yticks(range(T))
        ax.set_xticklabels(list(probe_text), rotation=90, fontsize=7)
        ax.set_yticklabels(list(probe_text), fontsize=7)
    fig.suptitle("First-block attention on a real prompt")
    fig.tight_layout(); plt.show()


---

## Extension: BPE Tokenization, Shared Raw-Text Split, and a Resume Phase

*(A self-contained follow-up: the sealed model above is untouched by anything below. BPE is fit on the shared training partition and compared on the same held-out raw text via unsmoothed bits-per-character and Char-KL.)*

### A From-Scratch BPE Tokenizer

Word-frequency-based training: count adjacent-symbol pairs weighted by frequency, and repeatedly merge the most frequent pair. Progress prints every 500 merges -- pure CPU work, so silence would otherwise look like a hang.

In [ ]:
import re
from collections import Counter

class SimpleBPE:
    def __init__(self):
        self.merges = []
        self.vocab = {}
        self.inv_vocab = {}

    def _word_to_symbols(self, word):
        return list(word) + ["</w>"]

    def train(self, text, vocab_size, verbose=True):
        words = re.findall(r"\S+|\s+", text)
        word_freq = Counter(words)
        splits = {w: self._word_to_symbols(w) for w in word_freq}

        base_chars = set()
        for w in word_freq:
            base_chars.update(self._word_to_symbols(w))
        self.vocab = {"<unk>": 0}
        for i, c in enumerate(sorted(base_chars), start=1):
            self.vocab[c] = i

        target_merges = vocab_size - len(self.vocab)
        pbar = tqdm(total=target_merges, disable=not verbose, desc="BPE merges")
        while len(self.vocab) < vocab_size:
            pair_counts = Counter()
            for w, freq in word_freq.items():
                symbols = splits[w]
                for i in range(len(symbols) - 1):
                    pair_counts[(symbols[i], symbols[i + 1])] += freq
            if not pair_counts:
                break
            best_pair, best_count = pair_counts.most_common(1)[0]
            if best_count < 2:
                break
            merged = best_pair[0] + best_pair[1]
            self.merges.append(best_pair)
            self.vocab[merged] = len(self.vocab)
            for w in list(splits.keys()):
                symbols, new_symbols, i = splits[w], [], 0
                while i < len(symbols):
                    if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == best_pair:
                        new_symbols.append(merged); i += 2
                    else:
                        new_symbols.append(symbols[i]); i += 1
                splits[w] = new_symbols
            pbar.update(1)
            pbar.set_postfix(vocab=len(self.vocab))
        pbar.close()

        self.inv_vocab = {i: t for t, i in self.vocab.items()}
        return self

    def _encode_word(self, word):
        symbols = self._word_to_symbols(word)
        for a, b in self.merges:
            merged = a + b
            new_symbols, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                    new_symbols.append(merged); i += 2
                else:
                    new_symbols.append(symbols[i]); i += 1
            symbols = new_symbols
        return symbols

    def encode(self, text):
        words = re.findall(r"\S+|\s+", text)
        cache = {}  # memoize per unique word -- real text repeats common words
                    # ("the", "of", ...) thousands of times; without this, encoding
                    # a multi-million-character corpus re-applies every merge rule
                    # to every occurrence instead of once per unique word (~15-50x slower)
        ids = []
        for w in words:
            if w not in cache:
                cache[w] = [self.vocab.get(sym, self.vocab["<unk>"]) for sym in self._encode_word(w)]
            ids.extend(cache[w])
        return ids

    def decode(self, ids):
        toks = [self.inv_vocab.get(i, "") for i in ids]
        return "".join(toks).replace("</w>", "")


BPE should compress a frequent word into fewer tokens than its character length.

In [ ]:
_probe = SimpleBPE().train("the committee decided " * 20, vocab_size=60, verbose=False)
_ids = _probe.encode("committee")
assert len(_ids) < len("committee")
print(f"'committee' (9 chars) -> {len(_ids)} tokens after training on repeated text")


### Train the Tokenizer on Training Text Only

Vocab size 4,000: smaller vocabularies tend to win on perplexity at some cost to output diversity. Merge rules and vocabulary entries are learned only from the shared raw-text training partition; validation text is encoded afterward without contributing frequency information.

In [ ]:
BPE_VOCAB_SIZE = 4000
bpe = SimpleBPE().train(train_text, vocab_size=BPE_VOCAB_SIZE)
print(f"BPE vocab: {len(bpe.vocab)} (fit on training text only)")

# Hash the learned merge order and ID-ordered vocabulary so the exact deterministic
# tokenizer used by a run can be identified without embedding it in every record.
_bpe_provenance = {
    "merges": bpe.merges,
    "vocab_by_id": sorted(bpe.vocab.items(), key=lambda item: item[1]),
}
bpe_tokenizer_sha256 = hashlib.sha256(
    json.dumps(_bpe_provenance, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
).hexdigest()

print("encoding train and validation partitions separately (cached per unique word)...")
_t0 = time.time()
bpe_train_ids = bpe.encode(train_text)
bpe_val_ids = bpe.encode(val_text)
print(f"train: {len(train_text):,} chars -> {len(bpe_train_ids):,} BPE tokens")
print(f"val:   {len(val_text):,} chars -> {len(bpe_val_ids):,} BPE tokens "
      f"in {time.time()-_t0:.1f}s")

unk_id = bpe.vocab["<unk>"]
val_unseen_chars = [c for c in val_text if c not in bpe.vocab]
val_unk_tokens = sum(token_id == unk_id for token_id in bpe_val_ids)
assert val_unk_tokens == len(val_unseen_chars)
print(f"validation fallback: {len(val_unseen_chars):,} characters -> "
      f"{val_unk_tokens:,} <unk> tokens; distinct={sorted(set(val_unseen_chars))!r}")

_roundtrip_cases = {
    "ordinary FOMC text": train_text[10000:10500],
    "whitespace": " \n\n ",
    "punctuation": "FOMC: rates, inflation; employment.",
}
for _label, _sample in _roundtrip_cases.items():
    assert all(c in bpe.vocab for c in _sample), f"{_label} contains an unrepresentable character"
    assert bpe.decode(bpe.encode(_sample)) == _sample, f"round-trip failed: {_label}"

_unseen_probe = val_unseen_chars[0] if val_unseen_chars else "☃"
assert _unseen_probe not in bpe.vocab
_unseen_ids = bpe.encode(_unseen_probe)
assert unk_id in _unseen_ids
assert bpe.decode(_unseen_ids) == "<unk>"
print("[OK] exact round trips: ordinary text, whitespace, punctuation")
print(f"[OK] unseen Unicode fallback: {_unseen_probe!r} -> '<unk>'")
print(f"[OK] tokenizer provenance SHA-256: {bpe_tokenizer_sha256}")


### BPE Train/Validation Sequences

The BPE model uses the same contiguous raw-text 90/10 boundary as the character model. Each partition is encoded separately, so validation text is never redistributed and sampled windows cannot cross artificial joins between nonadjacent chunks.

In [ ]:
bpe_train = torch.tensor(bpe_train_ids, dtype=torch.long)
bpe_val = torch.tensor(bpe_val_ids, dtype=torch.long)
assert len(bpe_train) == len(bpe_train_ids) and len(bpe_val) == len(bpe_val_ids)
print(f"BPE train: {len(bpe_train):,} tokens | val: {len(bpe_val):,} tokens")

def get_batch_bpe(split, block_size, batch_size, device=DEVICE):
    d = bpe_train if split == "train" else bpe_val
    ix = torch.randint(0, len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)


### Train a Matched-Architecture Model on BPE Tokens

The BPE model keeps the same Transformer depth, width, block architecture, training-step budget, and token-context budget. Its vocabulary-dependent tied embedding/output matrix is larger, and its tokens span more raw characters, so neither total parameter count nor raw-character context span is matched. Weight decay is 0.08; its effect remains a hypothesis because it was not isolated in a controlled ablation.

In [ ]:
import dataclasses
bpe_model_cfg = dataclasses.replace(model_cfg, vocab_size=len(bpe.vocab))
bpe_model = TinyTransformerLM(bpe_model_cfg).to(DEVICE)
bpe_n_params = sum(p.numel() for p in bpe_model.parameters())
print(f"BPE model parameters: {bpe_n_params:,}")

BPE_CKPT = CKPT_DIR / "bpe_transformer_best.pt"
bpe_train_cfg = dataclasses.replace(train_cfg, weight_decay=0.08)
bpe_initial_best_val = None
bpe_resume_best_val = None

if TRAIN:
    opt_bpe = torch.optim.AdamW(bpe_model.parameters(), lr=bpe_train_cfg.lr,
                                 weight_decay=bpe_train_cfg.weight_decay)
    bpe_history = {"step": [], "train_loss": [], "val_loss": [], "lr": [], "pre_clip_grad_norm": []}
    bpe_best_val = float("inf")
    pbar = tqdm(range(1, bpe_train_cfg.max_steps + 1))
    for step in pbar:
        lr = lr_at(step, bpe_train_cfg)
        for g in opt_bpe.param_groups: g["lr"] = lr
        xb, yb = get_batch_bpe("train", bpe_model_cfg.block_size, bpe_train_cfg.batch_size)
        _, loss = bpe_model(xb, yb)
        opt_bpe.zero_grad(); loss.backward()
        pre_clip_grad_norm = torch.nn.utils.clip_grad_norm_(bpe_model.parameters(), bpe_train_cfg.grad_clip)
        opt_bpe.step()
        if step % bpe_train_cfg.eval_interval == 0 or step == bpe_train_cfg.max_steps:
            tr = evaluate(bpe_model, "train", bpe_model_cfg.block_size, bpe_train_cfg.batch_size, bpe_train_cfg.eval_iters, get_batch_bpe)
            va = evaluate(bpe_model, "val", bpe_model_cfg.block_size, bpe_train_cfg.batch_size, bpe_train_cfg.eval_iters, get_batch_bpe)
            bpe_history["step"].append(step); bpe_history["train_loss"].append(tr); bpe_history["val_loss"].append(va)
            bpe_history["lr"].append(lr); bpe_history["pre_clip_grad_norm"].append(float(pre_clip_grad_norm))
            pbar.set_postfix(train=f"{tr:.3f}", val=f"{va:.3f}", lr=f"{lr:.1e}", pre_gnorm=f"{float(pre_clip_grad_norm):.2f}")
            if va < bpe_best_val:
                bpe_best_val = va
                save_checkpoint(bpe_model, bpe_model_cfg, va, BPE_CKPT)
    bpe_initial_best_val = bpe_best_val
    print(f"local BPE checkpoint (initial best val {bpe_best_val:.4f}): {BPE_CKPT}")
else:
    print("TRAIN is False -- BPE training is skipped; a local checkpoint will be loaded if present.")


### Resume Phase (BPE)

In [ ]:
if TRAIN:
    ckpt = torch.load(BPE_CKPT, map_location=DEVICE, weights_only=False)
    bpe_model.load_state_dict(ckpt["model_state_dict"])

    opt_resume = torch.optim.AdamW(bpe_model.parameters(), lr=BPE_RESUME_LR, weight_decay=0.0)
    patience_ctr = 0
    resume_history = {"step": [], "val_loss": []}

    pbar = tqdm(range(1, BPE_RESUME_STEPS + 1))
    for step in pbar:
        progress = step / BPE_RESUME_STEPS
        lr = BPE_RESUME_LR * 0.5 * (1 + math.cos(math.pi * progress))
        for g in opt_resume.param_groups: g["lr"] = lr
        xb, yb = get_batch_bpe("train", bpe_model_cfg.block_size, bpe_train_cfg.batch_size)
        _, loss = bpe_model(xb, yb)
        opt_resume.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(bpe_model.parameters(), 1.0)
        opt_resume.step()
        if step % RESUME_EVAL_INTERVAL == 0 or step == BPE_RESUME_STEPS:
            va = evaluate(bpe_model, "val", bpe_model_cfg.block_size, bpe_train_cfg.batch_size, RESUME_EVAL_ITERS, get_batch_bpe)
            resume_history["step"].append(step); resume_history["val_loss"].append(va)
            pbar.set_postfix(val=f"{va:.3f}", lr=f"{lr:.1e}")
            if va < bpe_best_val - EARLY_STOP_MIN_DELTA:
                bpe_best_val = va; patience_ctr = 0
                save_checkpoint(bpe_model, bpe_model_cfg, va, BPE_CKPT)
            else:
                patience_ctr += 1
                if patience_ctr >= EARLY_STOP_PATIENCE:
                    print(f"early stopping at step {step}: no improvement for {EARLY_STOP_PATIENCE} evals")
                    break
    bpe_resume_best_val = min(resume_history["val_loss"])
    print(f"BPE resume phase best val loss: {bpe_resume_best_val:.4f}")

if BPE_CKPT.exists():
    bpe_ckpt = torch.load(BPE_CKPT, map_location=DEVICE, weights_only=False)
    bpe_model.load_state_dict(bpe_ckpt["model_state_dict"])
    bpe_model.eval()
    bpe_checkpoint_val = bpe_ckpt["val_loss"]
    print(f"local BPE checkpoint loaded; stored best val CE {bpe_checkpoint_val:.4f}")
else:
    bpe_checkpoint_val = None
    print(f"no local checkpoint at {BPE_CKPT}; set TRAIN=True and run training to create it")

models_ready = CKPT_PATH.exists() and BPE_CKPT.exists()


### Sample Comparison

When both local checkpoints are available, this produces a direct side-by-side with `top_k=40` for both.

In [ ]:
def sample_bpe(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_k=None, top_p=None):
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=DEVICE)
    out = model.generate(idx, max_new_tokens, temperature=temperature, greedy=False, top_k=top_k, top_p=top_p)
    return tokenizer.decode(out[0].tolist())

sample_prompts = ["The Committee decided to ", "Meeting of the Federal Open Market Committee"]
char_samples, bpe_samples = {}, {}
if models_ready:
    for p in sample_prompts:
        print("=" * 92)
        print(f"PROMPT: {p!r}")
        print("-" * 92)
        c_out = sample(model, p, 150, temperature=0.7, top_k=40) if set(p) <= set(stoi) else "(char OOV)"
        b_out = sample_bpe(bpe_model, bpe, p, 150, temperature=0.7, top_k=40)
        char_samples[p], bpe_samples[p] = c_out, b_out
        show("char-level:", c_out)
        show("BPE:", b_out)
        print()
else:
    print("sample comparison skipped: local char and BPE checkpoints are both required")


### Char-KL: Character-Frequency Divergence

A cheap, training-free complement to bits-per-character: divergence between generated and real-corpus character-frequency distributions. Lower means the output's character statistics look more natural, independent of whether the words are meaningful.

In [ ]:
def char_kl_divergence(generated_text, reference_text):
    gen_counts = Counter(generated_text)
    ref_counts = Counter(reference_text)
    alphabet = set(gen_counts) | set(ref_counts)
    gen_total = sum(gen_counts.values())
    ref_total = sum(ref_counts.values())
    eps = 1e-10
    kl = 0.0
    for c in alphabet:
        p = gen_counts.get(c, 0) / gen_total
        q = ref_counts.get(c, 0) / ref_total
        if p > 0:
            kl += p * math.log((p + eps) / (q + eps))
    return kl

reference_sample = text[:200000]  # a real corpus slice as the reference distribution
if models_ready:
    char_kl_scores = {p: char_kl_divergence(char_samples[p], reference_sample) for p in sample_prompts}
    bpe_kl_scores = {p: char_kl_divergence(bpe_samples[p], reference_sample) for p in sample_prompts}

    for p in sample_prompts:
        print(f"{p!r}:")
        print(f"  char-level Char-KL: {char_kl_scores[p]:.4f}")
        print(f"  BPE        Char-KL: {bpe_kl_scores[p]:.4f}")
else:
    print("Char-KL skipped: local char and BPE checkpoints are both required")


### Fair Comparison: Unsmoothed Held-Out Bits Per Character

Optimization diagnostics above retain label smoothing. The final information-theoretic comparison instead sums ordinary unsmoothed next-token NLL over deterministic overlapping windows from the same held-out raw-text partition, scoring every target exactly once, then divides by the number of held-out raw characters and `log(2)`. Both models use the same `block_size` in tokens; because BPE tokens span more raw characters, this compares tokenizer/model systems under an equal token-context budget, not an equal raw-character context span.

In [ ]:
def bits_per_character(total_nll, raw_characters_scored):
    return total_nll / (raw_characters_scored * math.log(2))

if models_ready:
    # Reload both best local checkpoints before the deterministic final evaluator.
    char_ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(char_ckpt["model_state_dict"])
    bpe_ckpt = torch.load(BPE_CKPT, map_location=DEVICE, weights_only=False)
    bpe_model.load_state_dict(bpe_ckpt["model_state_dict"])

    char_eval_ids = torch.cat([train_data[-1:], val_data])
    bpe_eval_ids = torch.cat([bpe_train[-1:], bpe_val])
    char_eval_stride = max(1, model_cfg.block_size // 2)
    bpe_eval_stride = max(1, bpe_model_cfg.block_size // 2)
    char_total_nll, char_raw_chars = evaluate_unsmoothed_nll(
        model, char_eval_ids, model_cfg.block_size, len(val_text), stride=char_eval_stride
    )
    bpe_total_nll, bpe_raw_chars = evaluate_unsmoothed_nll(
        bpe_model, bpe_eval_ids, bpe_model_cfg.block_size, len(val_text), stride=bpe_eval_stride
    )
    assert char_raw_chars == bpe_raw_chars == len(val_text)

    char_bpc = bits_per_character(char_total_nll, char_raw_chars)
    bpe_bpc = bits_per_character(bpe_total_nll, bpe_raw_chars)
    print("label-smoothed optimization diagnostics (not BPC):")
    print(f"  char checkpoint best val CE: {char_checkpoint_val:.4f}")
    print(f"  BPE  checkpoint best val CE: {bpe_checkpoint_val:.4f}")
    print("unsmoothed deterministic held-out metrics:")
    print(f"  char: total NLL {char_total_nll:.2f} over {char_raw_chars:,} raw chars -> "
          f"{char_bpc:.4f} bits/char")
    print(f"  BPE : total NLL {bpe_total_nll:.2f} over {bpe_raw_chars:,} raw chars -> "
          f"{bpe_bpc:.4f} bits/char")

    if TRAIN:
        char_tokenizer_sha256 = hashlib.sha256(
            json.dumps(sorted(stoi.items()), ensure_ascii=False, separators=(",", ":")).encode("utf-8")
        ).hexdigest()
        char_run_path = save_final_run(
            model_name="char",
            tokenizer={"type": "character", "vocabulary_size": vocab_size,
                       "vocabulary_sha256": char_tokenizer_sha256},
            cfg_model=model_cfg, cfg_train=train_cfg,
            resume_config={"max_steps": CHAR_RESUME_STEPS, "learning_rate": CHAR_RESUME_LR,
                           "weight_decay": 0.0, "eval_interval": RESUME_EVAL_INTERVAL,
                           "eval_iters": RESUME_EVAL_ITERS,
                           "early_stop_patience": EARLY_STOP_PATIENCE,
                           "early_stop_min_delta": EARLY_STOP_MIN_DELTA},
            split_metadata=split_metadata, checkpoint_path=CKPT_PATH, parameter_count=n_params,
            initial_best_smoothed_val_ce=char_initial_best_val,
            resume_best_smoothed_val_ce=char_resume_best_val,
            final_unsmoothed_nll=char_total_nll, final_bpc=char_bpc,
            evaluation={"method": "deterministic_unsmoothed_sliding_window",
                        "block_size": model_cfg.block_size, "stride": char_eval_stride,
                        "label_smoothing": 0.0, "reduction": "sum"},
            corpus_sha256=corpus_sha256,
        )
        bpe_run_path = save_final_run(
            model_name="bpe",
            tokenizer={"type": "bpe", "vocabulary_size": len(bpe.vocab),
                       "fitted_on_train_only": True,
                       "merge_and_vocabulary_sha256": bpe_tokenizer_sha256,
                       "unknown_validation_character_count": len(val_unseen_chars)},
            cfg_model=bpe_model_cfg, cfg_train=bpe_train_cfg,
            resume_config={"max_steps": BPE_RESUME_STEPS, "learning_rate": BPE_RESUME_LR,
                           "weight_decay": 0.0, "eval_interval": RESUME_EVAL_INTERVAL,
                           "eval_iters": RESUME_EVAL_ITERS,
                           "early_stop_patience": EARLY_STOP_PATIENCE,
                           "early_stop_min_delta": EARLY_STOP_MIN_DELTA},
            split_metadata=split_metadata, checkpoint_path=BPE_CKPT, parameter_count=bpe_n_params,
            initial_best_smoothed_val_ce=bpe_initial_best_val,
            resume_best_smoothed_val_ce=bpe_resume_best_val,
            final_unsmoothed_nll=bpe_total_nll, final_bpc=bpe_bpc,
            evaluation={"method": "deterministic_unsmoothed_sliding_window",
                        "block_size": bpe_model_cfg.block_size, "stride": bpe_eval_stride,
                        "label_smoothing": 0.0, "reduction": "sum"},
            corpus_sha256=corpus_sha256,
        )
        print(f"final run records: {char_run_path} | {bpe_run_path}")
    else:
        print("TRAIN=False: final metrics evaluated from local checkpoints; no new run records "
              "written because phase-level training provenance is unavailable.")

    print()
    if bpe_bpc < char_bpc:
        print(f"BPE wins on the fair metric: {bpe_bpc:.4f} < {char_bpc:.4f} bits/char "
              f"({100*(1-bpe_bpc/char_bpc):.1f}% lower)")
    else:
        print(f"Char-level wins on the fair metric: {char_bpc:.4f} < {bpe_bpc:.4f} bits/char")
else:
    print("final NLL/BPC evaluation skipped: local char and BPE checkpoints are both required")


### Extension Conclusion

BPE achieves **0.9828 BPC** versus **1.2294 BPC** for the character model, a reduction of approximately **20.1%** under the corrected methodology. The comparison uses one shared contiguous raw-text split, fits BPE on training text only, and holds Transformer depth, width, block size, and token-context budget constant. Label-smoothed validation CE remains an optimization diagnostic; the final tokenizer comparison is deterministic unsmoothed held-out NLL normalized by raw validation characters.

The systems do not have equal parameter counts or equal raw-character context: vocabulary-dependent parameters are 5,756,928 for BPE versus 4,759,040 for character-level, and BPE tokens span more characters. Weight decay remains a hypothesis rather than a causal explanation because it was not isolated in an ablation. Char-KL is a secondary sample statistic, not the basis of the result.

## Interpretation

- **Character model:** final deterministic unsmoothed held-out performance is **1.2294 BPC** (NLL 573,632.875).
- **BPE model:** final performance is **0.9828 BPC** (NLL 458,539.46875), approximately **20.1% lower** than character-level under the shared-split evaluation.
- **Training stability:** best label-smoothed validation CE improved from 1.261518 to 1.237127 for character-level and from 2.621752 to 2.551549 for BPE, with no obvious divergence. CE diagnoses optimization; it is not the final tokenizer-comparison metric.
- **Gradient norms:** logged values are total norms measured before clipping; bounded traces indicate that clipping was available as a guardrail, not that every logged value is a post-clipping norm.
- **Generation quality:** samples capture FOMC names, procedural structure, and policy phrasing, but long-form coherence remains limited at this model and data scale.

## Limitations and Future Work

- Results use one seed (`seed=1`); there is no multi-seed uncertainty estimate.
- Vocabulary-dependent parameter counts and effective raw-text context differ even though depth, width, block size, and token-context budget are shared.
- Seven validation characters are unseen in BPE training and map to `<unk>`.
- The corpus has a documented coverage gap for pre-2010 minutes because those files use a legacy URL scheme.
- Larger models and controlled ablations, including weight decay and context/parameter matching, remain future work.

## Total Runtime

Fresh Colab execution on **NVIDIA L4**: **33 min 52 s** (`TRAIN=True`, Run All).

In [ ]:
elapsed_total = time.time() - NOTEBOOK_START
mins, secs = divmod(elapsed_total, 60)
print(f"total notebook runtime: {int(mins)} min {secs:.0f} s")
